# Algoritmo de previsão de vitórias
* Uma equipe profissional de LOL juntou uma base de dados e quer desenvolver um algoritmo capaz de predizer derrotas e vitórias de uma equipe para identificar quais aspectos do jogo são mais determinantes na performance.
## Paleta de cores para a equipe
* #FF6F61 - Vermelho
* #6F2C91 - Roxo
* #88B04B - Verde
* #F7CAC9 - Salmão
* #92A8D1 - Azul claro

In [2]:
# Biblioteca de manipulação de dados
import pandas as pd
import numpy as np
from scipy.stats import zscore
from scipy.stats import randint
from scipy.stats import loguniform, uniform

# Biblioteca de visualização de dados
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# Carregando base de dados
df = pd.read_csv('../src/data/raw/Base_LOL.csv')

df.head()

,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,...,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
0,4519157822,0,28,2,1,9,6,11,0,0,...,0,16567,6.8,17047,197,55,-643,8,19.7,1656.7
1,4523371949,0,12,1,0,5,5,5,0,0,...,1,17620,6.8,17438,240,52,2908,1173,24.0,1762.0
2,4521474530,0,15,0,0,7,11,4,1,1,...,0,17285,6.8,17254,203,28,1172,1033,20.3,1728.5
3,4524384067,0,43,1,0,4,5,5,1,0,...,0,16478,7.0,17961,235,47,1321,7,23.5,1647.8
4,4436033771,0,75,4,0,6,6,6,0,0,...,0,17404,7.0,18313,225,67,1004,-230,22.5,1740.4


In [4]:
# Verificação de colunas e dados
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9879 entries, 0 to 9878
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   gameId                        9879 non-null   int64  
 1   blueWins                      9879 non-null   int64  
 2   blueWardsPlaced               9879 non-null   int64  
 3   blueWardsDestroyed            9879 non-null   int64  
 4   blueFirstBlood                9879 non-null   int64  
 5   blueKills                     9879 non-null   int64  
 6   blueDeaths                    9879 non-null   int64  
 7   blueAssists                   9879 non-null   int64  
 8   blueEliteMonsters             9879 non-null   int64  
 9   blueDragons                   9879 non-null   int64  
 10  blueHeralds                   9879 non-null   int64  
 11  blueTowersDestroyed           9879 non-null   int64  
 12  blueTotalGold                 9879 non-null   int64  
 13  blueAvgLevel  

In [5]:
# Métricas básicas das variáveis
pd.set_option('display.max_columns', None)
df.describe()

,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,blueTowersDestroyed,blueTotalGold,blueAvgLevel,blueTotalExperience,blueTotalMinionsKilled,blueTotalJungleMinionsKilled,blueGoldDiff,blueExperienceDiff,blueCSPerMin,blueGoldPerMin,redWardsPlaced,redWardsDestroyed,redFirstBlood,redKills,redDeaths,redAssists,redEliteMonsters,redDragons,redHeralds,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
count,9.879000e+03,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000,9879.000000
mean,4.500084e+09,0.499038,22.288288,2.824881,0.504808,6.183925,6.137666,6.645106,0.549954,0.361980,0.187974,0.051422,16503.455512,6.916004,17928.110133,216.699565,50.509667,14.414111,-33.620306,21.669956,1650.345551,22.367952,2.723150,0.495192,6.137666,6.183925,6.662112,0.573135,0.413098,0.160036,0.043021,16489.041401,6.925316,17961.730438,217.349226,51.313088,-14.414111,33.620306,21.734923,1648.904140
std,2.757328e+07,0.500024,18.019177,2.174998,0.500002,3.011028,2.933818,4.064520,0.625527,0.480597,0.390712,0.244369,1535.446636,0.305146,1200.523764,21.858437,9.898282,2453.349179,1920.370438,2.185844,153.544664,18.457427,2.138356,0.500002,2.933818,3.011028,4.060612,0.626482,0.492415,0.366658,0.216900,1490.888406,0.305311,1198.583912,21.911668,10.027885,2453.349179,1920.370438,2.191167,149.088841
min,4.295358e+09,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10730.000000,4.600000,10098.000000,90.000000,0.000000,-10830.000000,-9333.000000,9.000000,1073.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11212.000000,4.800000,10465.000000,107.000000,4.000000,-11467.000000,-8348.000000,10.700000,1121.200000
25%,4.483301e+09,0.000000,14.000000,1.000000,0.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.000000,0.000000,15415.500000,6.800000,17168.000000,202.000000,44.000000,-1585.500000,-1290.500000,20.200000,1541.550000,14.000000,1.000000,0.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.000000,0.000000,15427.500000,6.800000,17209.500000,203.000000,44.000000,-1596.000000,-1212.000000,20.300000,1542.750000
50%,4.510920e+09,0.000000,16.000000,3.000000,1.000000,6.000000,6.000000,6.000000,0.000000,0.000000,0.000000,0.000000,16398.000000,7.000000,17951.000000,218.000000,50.000000,14.000000,-28.000000,21.800000,1639.800000,16.000000,2.000000,0.000000,6.000000,6.000000,6.000000,0.000000,0.000000,0.000000,0.000000,16378.000000,7.000000,17974.000000,218.000000,51.000000,-14.000000,28.000000,21.800000,1637.800000
75%,4.521733e+09,1.000000,20.000000,4.000000,1.000000,8.000000,8.000000,9.000000,1.000000,1.000000,0.000000,0.000000,17459.000000,7.200000,18724.000000,232.000000,56.000000,1596.000000,1212.000000,23.200000,1745.900000,20.000000,4.000000,1.000000,8.000000,8.000000,9.000000,1.000000,1.000000,0.000000,0.000000,17418.500000,7.200000,18764.500000,233.000000,57.000000,1585.500000,1290.500000,23.300000,1741.850000
max,4.527991e+09,1.000000,250.000000,27.000000,1.000000,22.000000,22.000000,29.000000,2.000000,1.000000,1.000000,4.000000,23701.000000,8.000000,22224.000000,283.000000,92.000000,11467.000000,8348.000000,28.300000,2370.100000,276.000000,24.000000,1.000000,22.000000,22.000000,28.000000,2.000000,1.000000,1.000000,2.000000,22732.000000,8.200000,22269.000000,289.000000,92.000000,10830.000000,9333.000000,28.900000,2273.200000


# Termos do jogo
* O jogo é de tipo competitivo e de invasão;
* Gold é a moeda da partida. Obtida a partir de ações na partida. Gasta para melhorias durante a partida;
* Experience é uma medida de evolução do personagem escolhido, fortalecendo-o a partir de ações tomadas durante a partida;
* Wards placed: Sentinelas posicionadas para dar visão do mapa. Impacto: aumenta o controle de território e previne emboscadas.
* Wards destroyed: Sentinelas inimigas eliminadas. Impacto: reduz a visão do adversário, abrindo espaço para jogadas arriscadas.
* Elite monsters: Criaturas poderosas como Dragão, Arauto e Barão. Impacto: concedem bônus fortes que podem mudar o rumo da partida.
* Dragons: Monstros elementais que dão buffs permanentes à equipe. Impacto: acumulados, fortalecem muito o time; Alma do Dragão é decisiva.
* Heralds: Arauto do Vale. Impacto: ajuda a destruir torres rapidamente, acelerando o avanço no mapa.
* Towers: Estruturas defensivas que protegem o mapa. Impacto: cada torre destruída abre espaço e dá ouro, aproximando a equipe da vitória.
* Minions: Tropas que avançam nas rotas. Impacto: fonte principal de ouro e experiência, além de pressionar o mapa.
* Jungle minions: Criaturas da selva. Impacto: fornecem ouro, experiência e buffs temporários (como azul e vermelho).
* CS per minute: “Creep Score” por minuto, ou seja, quantos minions/jungle minions o jogador abate em média por minuto. Impacto: indicador de eficiência na coleta de recursos; quanto maior, mais forte o jogador fica em ouro e itens.

# Tratamento e análise de dados
* Apesar de não conter dados nulos, há a possibilidade de outliers;
* Se houver, deverão ser tratados

In [6]:
# Uso de z-score para tratamento de possíveis outliers
z_score = df.apply(zscore)
mask = ~(z_score.abs() > 3).any(axis=1)
df_limpo = df[mask]

In [7]:
df_limpo.describe()

,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,blueTowersDestroyed,blueTotalGold,blueAvgLevel,blueTotalExperience,blueTotalMinionsKilled,blueTotalJungleMinionsKilled,blueGoldDiff,blueExperienceDiff,blueCSPerMin,blueGoldPerMin,redWardsPlaced,redWardsDestroyed,redFirstBlood,redKills,redDeaths,redAssists,redEliteMonsters,redDragons,redHeralds,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
count,7.924000e+03,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.0,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.0,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000
mean,4.500883e+09,0.501010,19.699016,2.697501,0.502524,6.012998,5.970217,6.436648,0.532181,0.361055,0.171126,0.0,16386.115851,6.922009,17946.042277,217.434377,50.749243,1.742428,-40.449142,21.743438,1638.611585,19.892226,2.605881,0.497476,5.970217,6.012998,6.437405,0.559187,0.412797,0.146391,0.0,16384.373423,6.932408,17986.491418,218.272211,51.477915,-1.742428,40.449142,21.827221,1638.437342
std,2.567898e+07,0.500031,10.235367,1.689964,0.500025,2.789191,2.733365,3.728194,0.616652,0.480337,0.376643,0.0,1351.430165,0.279241,1098.927015,20.833869,9.551093,2115.213407,1720.530448,2.083387,135.143016,10.623560,1.648844,0.500025,2.733365,2.789191,3.751729,0.619352,0.492368,0.353520,0.0,1325.169249,0.278480,1094.993966,20.774028,9.709273,2115.213407,1720.530448,2.077403,132.516925
min,4.417370e+09,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,12682.000000,6.200000,14483.000000,152.000000,21.000000,-6717.000000,-5585.000000,15.200000,1268.200000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,12626.000000,6.200000,14418.000000,153.000000,22.000000,-6547.000000,-5575.000000,15.300000,1262.600000
25%,4.483891e+09,0.000000,14.000000,1.000000,0.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.000000,0.0,15406.500000,6.800000,17200.750000,203.000000,44.000000,-1449.000000,-1209.250000,20.300000,1540.650000,14.000000,1.000000,0.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.000000,0.0,15418.750000,6.800000,17257.500000,204.000000,44.000000,-1446.750000,-1143.000000,20.400000,1541.875000
50%,4.511148e+09,1.000000,16.000000,3.000000,1.000000,6.000000,6.000000,6.000000,0.000000,0.000000,0.000000,0.0,16331.000000,7.000000,17953.500000,218.000000,50.000000,1.500000,-36.000000,21.800000,1633.100000,16.000000,2.000000,0.000000,6.000000,6.000000,6.000000,0.000000,0.000000,0.000000,0.0,16313.000000,7.000000,17973.000000,219.000000,52.000000,-1.500000,36.000000,21.900000,1631.300000
75%,4.521821e+09,1.000000,19.000000,4.000000,1.000000,8.000000,8.000000,9.000000,1.000000,1.000000,0.000000,0.0,17303.000000,7.200000,18700.000000,232.000000,56.000000,1446.750000,1143.000000,23.200000,1730.300000,19.000000,4.000000,1.000000,8.000000,8.000000,9.000000,1.000000,1.000000,0.000000,0.0,17275.750000,7.200000,18737.250000,233.000000,58.000000,1449.000000,1209.250000,23.300000,1727.575000
max,4.527960e+09,1.000000,76.000000,9.000000,1.000000,15.000000,14.000000,18.000000,2.000000,1.000000,1.000000,0.0,21095.000000,7.800000,21508.000000,281.000000,80.000000,6547.000000,5575.000000,28.100000,2109.500000,77.000000,9.000000,1.000000,14.000000,15.000000,18.000000,2.000000,1.000000,1.000000,0.0,20920.000000,7.800000,21517.000000,282.000000,81.000000,6717.000000,5585.000000,28.200000,2092.000000


In [8]:
# Correlação entre blueCSPerMin e blueWins
cs_w = df_limpo.groupby(['blueCSPerMin','blueWins'])['blueCSPerMin'].size().reset_index(name='count')

fig = px.histogram(cs_w,
                   x='blueCSPerMin',
                   y='count',
                   color='blueWins',
                   color_discrete_map={0:'#FF6F61', 1:'#6F2C91'})
fig.update_layout(
    template='plotly_white',
    title = 'Cs Per min X Vitórias',
    xaxis_title='Cs Per Min',
    yaxis_title='Contagem',
    barmode='group'
)
fig.show()

In [9]:
# Correlação blueDeaths e blueWins
death_w = df_limpo.groupby(['blueDeaths','blueWins'])['blueDeaths'].size().reset_index(name='count')

fig = px.histogram(death_w,
                   x='blueDeaths',
                   y='count',
                   color='blueWins',
                   color_discrete_map={0:'#FF6F61', 1:'#6F2C91'},
                   nbins=12)
fig.update_layout(
    template='plotly_white',
    title = 'Blue team Deaths X Vitórias',
    xaxis_title='Blue team Deaths',
    yaxis_title='Contagem',
    barmode='group'
)
fig.show()

In [10]:
# Correlação entre blueAvgLevel e blueWins
lvl_w = df_limpo.groupby(['blueAvgLevel','blueWins'])['blueAvgLevel'].size().reset_index(name='count')

fig = px.histogram(lvl_w,
                   x='blueAvgLevel',
                   y='count',
                   color='blueWins',
                   color_discrete_map={0:'#FF6F61', 1:'#6F2C91'},
                   nbins=10)
fig.update_layout(
    template='plotly_white',
    title = 'Blue Avg Level X Vitórias',
    xaxis_title='Blue Avg Level',
    yaxis_title='Contagem',
    barmode='group'
)
fig.show()

In [11]:
# Correlação entre redEliteMonsters e BlueWins
rem_w = df_limpo.groupby(['redEliteMonsters','blueWins'])['redEliteMonsters'].size().reset_index(name='count')

fig = px.histogram(rem_w,
                   x='redEliteMonsters',
                   y='count',
                   color='blueWins',
                   color_discrete_map={0:'#FF6F61', 1:'#6F2C91'},
                   nbins=3)
fig.update_layout(
    template='plotly_white',
    title = 'Red elite monsters X Vitórias',
    xaxis_title='Número de monstros',
    yaxis_title='Contagem',
    barmode='group'
)
fig.show()

In [12]:
# Correlação entre blueTotalMinionsKilled e blueWins
min_w = df_limpo.groupby(['blueTotalMinionsKilled','blueWins'])['blueTotalMinionsKilled'].size().reset_index(name='count')

fig = px.histogram(min_w,
                   x='blueTotalMinionsKilled',
                   y='count',
                   color='blueWins',
                   color_discrete_map={0:'#FF6F61', 1:'#6F2C91'},
                   nbins=20)
fig.update_layout(
    template='plotly_white',
    title = 'Minions mortos pelo time azul X Vitórias',
    xaxis_title='Número de minions',
    yaxis_title='Contagem',
    barmode='group'
)
fig.show()

In [13]:
# Gráfico de correlação de variáveis
'''
Como são muitas colunas vou dividir em 2 gráficos, mantendo a variável alvo
'''
df_limpo = df_limpo.drop(columns='gameId')
df_blue = df_limpo.drop(columns= df.columns[21:])

df_red = df_limpo.drop(columns=df.columns[2:21])

corr_blue = df_blue.corr()
color_scale = [[0, '#6F2C91'],[1, '#88B04B']]

fig = px.imshow(
    corr_blue,
    text_auto=True,
    aspect='auto',
    color_continuous_scale=color_scale,
)
fig.update_layout(
    title='Gráfico de correlação de variáveis da equipe azul',
    template='plotly_white',
)
fig.show()

In [14]:
corr_red = df_red.corr()
color_scale = [[0, '#6F2C91'],[1, '#88B04B']]

fig = px.imshow(
    corr_red,
    text_auto=True,
    aspect='auto',
    color_continuous_scale=color_scale,
)
fig.update_layout(
    title='Gráfico de correlação de variáveis da equipe vermelha',
    template='plotly_white',
)
fig.show()

## Resultados das análises
* Todas as estatísticas da equipe vermelha atuam em proporção inversa de correlação com as estatísticas da equipe azul. Logo, para a construção do modelo, se faz desnecessário utilizar as variáveis das duas equipes em conjunto;
* A equipe azul parece dominar a quantia de partidas ganhas a partir da contagem de 22 CS por minuto;
* Quando a equipe azul acumula 6 mortes ou mais da equipe, os adversários começam a obter mais vitórias;
* A partir da média de level 7 dos heróis da equipe azul, esta começa a sobrepujar os adversários em vitórias;
* A equipe com mais monstros de elite tende a vencer mais partidas;
* A equipe azul começa a ter um domínio maior das partidas quando alcança o total de 220 minions abatidos;
* Os gráficos de correlação de variáveis indicaram que nenhuma variável demonstrou forte correlação negativa ou positiva para justificar uma seleção individual para a modelagem.

In [16]:
# Uso do DF_limpo para a modelagem
df_limpo.to_csv('LOL_limpo.csv', index=False)